In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency


regions = ["H1", "H2", "L1", "L2", "L3"]
seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]
len_regions = ["LEN_H1", "LEN_H2", "LEN_L1", "LEN_L2", "LEN_L3"]
meta_cols = ["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species"]


ab_ag_scalop = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset = seq_regions)
    #.dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions) 
)

antigen_counts = ab_ag_scalop["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
ab_ag_scalop = ab_ag_scalop[ab_ag_scalop["antigen_name"].isin(antigen_counts[antigen_counts >= 10].index)]# Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt

In [2]:
for cf_region, seq_region in zip(cf_regions, seq_regions):    
    print(f"Cluster in {cf_region}: {sorted(ab_ag_scalop[cf_region].astype(str).unique().tolist())}")
    print(f"Längen in {cf_region}: {sorted(ab_ag_scalop[seq_region].astype(str).map(len).unique().tolist())}\n")

Cluster in CF_H1: ['H1-7-A', 'H1-7-B', 'H1-7-C', 'H1-7-D', 'H1-8-A', 'H1-8-B', 'H1-9-A', 'H1-9-B', 'nan']
Längen in CF_H1: [4, 5, 6, 7, 8, 9, 13, 16]

Cluster in CF_H2: ['H2-5-A', 'H2-6-A', 'H2-6-B', 'H2-6-C', 'H2-6-D', 'H2-6-E', 'H2-8-A', 'nan']
Längen in CF_H2: [4, 5, 6, 7, 8, 10, 11, 16]

Cluster in CF_L1: ['L1-10-A', 'L1-11-A', 'L1-11-B', 'L1-12-A', 'L1-12-B', 'L1-12-C', 'L1-13-A', 'L1-13-B', 'L1-13-C', 'L1-14-A', 'L1-14-B', 'L1-14-C', 'L1-15-A', 'L1-16,17-A', 'nan']
Längen in CF_L1: [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

Cluster in CF_L2: ['L2-7-A', 'nan']
Längen in CF_L2: [7, 11, 12]

Cluster in CF_L3: ['L3-10,11-A', 'L3-10-A', 'L3-10-B', 'L3-11-A', 'L3-5-A', 'L3-8-A', 'L3-9,10-A', 'L3-9-A', 'nan']
Längen in CF_L3: [5, 6, 8, 9, 10, 11, 12, 13]



In [3]:
# Before clustering, Scalop groups CDR-sequences by length. However, some sequences in our dataset have a length, for which Scalop cannot assign a cluster.
# These sequences are unusually long or short (e.g. für CDR-H1: lengths 4, 5, 6, 12, 13, 16 do not yield any clusters, lenghts 7 and 8 do yield clusters) and are rare in our dataset as they are outliers.
# Proportion tests will be conducted only on length-groups, for which Scalop can assign clusters. Sequences in those length-groups, which were not assigned a cluster (nan) will be considered as cluster "others"
# in their respective length-group. Length groups, that did not yield any clusters will be discarded and not analyzed further.

In [4]:
regions_dict = {}

for region, seq_region, cf_region, len_region in zip(regions, seq_regions, cf_regions, len_regions):

    # DataFrame für jede Region, der alle Sequenzen nach Längengruppe und Cluster sortiert

    df = ab_ag_scalop[meta_cols + [seq_region, cf_region]].copy() # Erstellt eue Kopie von df für jede Region, damit nicht jede Region den originalen df überschreibt
    df[len_region] = df[seq_region].str.len() # Neue Spalte für die Sequenzlänge
    df[cf_region] = df[cf_region].fillna("others") # Ersetzt fehlende Cluster durch "others" 
    df = df.sort_values(by=[len_region, cf_region]) # Sortiert nach Länge und Cluster


    # DataFrame für jede Region aufteilen in DataFrames für jede Längengruppe und filtern nach Längengruppen mit gültigen Cluster-Zuordnungen durch Scalop

    valid_lens = (
    df[df[cf_region] != "others"] # nur Zeilen, in denen es gültige Cluster-Zuordnungen durch Scalop gibt
    [len_region].unique().tolist() # Sequenzlängen, die in den gültigen Zeilen vorkommen, aus der Spalte zur Sequenzlänge extrahieren
    )

    len_groups_dict = {} # Dictionary initialisieren für die DataFrames aller gültigen Längen-Gruppen
    for len_group in valid_lens:
        df_len = df[df[len_region] == len_group] # Zeilen aus dem großen DataFrame filtern, die zur gleichen, gültigen Längengruppe gehören
        len_groups_dict[len_group] = df_len # gefilterten DataFrame in das Dictionary aufnehmen

    regions_dict[region] = len_groups_dict # Dictionary für DataFrames derselben Region in ein äußeres Dictionary anlegen

In [5]:
def permutation_test(contingency, n_permutations, seed=None):
    np.random.seed(seed)
    # Setzt Startpunkt der Generation von Zufallszahlen fest, damit Ergebnisse bei Wiederholung vergleichbar sind

    # Beobachtete Teststatistik
    chi2_obs, _, _, _ = chi2_contingency(contingency)
    # chi2_contingency führt einen klassischen Chi2-Unabhängigkeitstest basierend auf der beobachteten Kontingenztabelle durch, d.h:
    # - berechnet beobachtete Teststatistik Chi2 = (Oij - Eij)^2/Eij
    # - vergleicht beobachteten Chi2-Wert mit theoretischer Chi2-Verteilung und gibt p-Wert (entfällt, weil Annahme der theoretischen Chi2-Verteilung ungültig)
    # - gibt dof der Kontingenztabelle bei festen Randwerten aus (hier nicht benötigt)
    # - gibt theoretische Kontingenztabelle mit den Erwartungswerten Eij = ((Summe der Zeile i) * (Summe der Spalte j))/Gesamtsumme aus (hier nicht benötigt)

    # Einträge der Kontingenztabelle als Liste
    data = contingency.stack().reset_index().values.tolist()
    # .stack(): Erstellt aus der Kontingenztabelle eine pd.Series (Index: (Cluster i, Antigen j), Wert: Oij) DATENTYP
    # .reset_index(): Erstellt aus pd.Series mit zweiwertigem Index und einem Wert eine Tabelle mit 3 Spalten (Cluster i, Antigen j, Oij) DATENTYP
    # .values(): Erstellt einen np.ndarray aus der Tabelle
    # .tolist(): Erstellt aus der np.ndarray eine Liste an Listen

    rows = contingency.index.tolist() # Liste der Reihennamen der Kontingenztabelle (Cluster), jedes Cluster kommt 1x vor
    cols = contingency.columns.tolist() # Liste der Spaltennamen der Kontingentabelle (Antigene), jedes Antigen kommt 1x vor
    
    values = []
    groups = []
    for row, col, count in data: # Für jedes Feld in der Kontingenztabelle (Feld wird definiert durch row (Cluster), col (Antigen) und count)
        for _ in range(count):
            values.append(col) # values: Liste an Antigenen, jedes Antigen kommt count mal vor durch die Schleife
            groups.append(row) # groups: Liste an Clustern, jedes Cluster kommt count mal vor durch die Schleife

    # Empirischen p-Wert berechnen
    greater_equal_count = 0 # Zähler initialisieren, wie oft ein permutierter Chi2-Wert mind. so groß ist wie der beobachtete Chi2-Wert
    for _ in range(n_permutations):
        shuffled = np.random.permutation(values) # Erzeuge eine zufällige Permutation der beobachteten Antigen-Häufigkeiten
        shuffled_table = pd.crosstab(groups, shuffled) # Erstelle eine Kontingenztabelle aus den permutierten Antigen-Häufigkeiten und den beobachteten Cluster-Häufigkeiten
        shuffled_table = shuffled_table.reindex(index=rows, columns=cols, fill_value=0)
        # Falls durch Permutation manche Antigene gar nicht mehr vorkommen, entfällt die Spalte nicht in der Kontingenztabelle, sondern die Felder bekommen count = 0
        chi2_perm, _, _, _ = chi2_contingency(shuffled_table) # wie oben, nur Chi2-Wert ist relevant

        if chi2_perm >= chi2_obs:
            greater_equal_count += 1 # Anzahl an Permutationen, die einen Chi2-Wert ergeben, der den beobachteten Chi2-Wert übertrifft

    p_value = greater_equal_count / n_permutations # empirischer p-Wert: Wahrscheinlichkeit, dass eine Permutation einen Chi2-Wert ergibt, der den beobachteten Chi2-Wert übertrifft
    return chi2_obs, p_value


In [ ]:
# Chi2-Permutationstest (Unabhängigkeitstest) für Vergleich der Cluster untereinander innerhalb derselben Längengruppen derselben CDR-Regionen

for region, cf_region in zip(regions, cf_regions):
    for len_group in regions_dict[region]:

        df = regions_dict[region][len_group]

        contingency = pd.crosstab(df[cf_region], df["antigen_name"])
        chi2_stat, p_value = permutation_test(contingency, n_permutations = 1000)

        print(f"{region} - Länge {len_group}: Chi² = {chi2_stat:.2f}, p = {p_value:.4f}")
        #print(contingency)


H1 - Länge 7: Chi² = 44.19, p = 0.5750


In [ ]:
# Chi2-Permutationstest (Unabhängigkeitstest)für Vergleich der Cluster untereiander innerhalb derselben CDR-Regionen (nicht getrennt nach Längengruppen)

for region, cf_region in zip(regions, cf_regions):
    
    df = pd.concat(regions_dict[region].values(), ignore_index=True)
    
    contingency = pd.crosstab(df[cf_region], df["antigen_name"])
    chi2_stat, p_value = permutation_test(contingency, n_permutations = 1000)
    
    print(f"{region} - Gesamte Region: Chi² = {chi2_stat:.2f}, df = {dof}, p = {p_value:.4f}")
    #print(contingency)